# 🛡️ Kaggle Hallucination Detection & Risk Assessment POC

This notebook evaluates various hallucination-detection approaches for a RAG-based support bot.
It runs fully on GPU (if available) and compares: 
1. **LettuceDetect**
2. **Vectara HHEM-2.1-Open**
3. **MiniCheck-Flan-T5-Large** (Sentence-level)
4. **Zero-Shot LLM Judge** (Llama 3.2)
5. **Few-Shot LLM Judge** (Llama 3.2 with adjacent-capability worked examples)
6. **Tiered/Composite Pipeline** (Relevance Gating + MiniCheck + LLM Judge Escalation)

---

In [ ]:
# ============================================================
# 🚀 Environment and GPU Check
# ============================================================
import torch
import os

print("=== Hardware Context ===")
device_available = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if device_available else "CPU"
print(f"CUDA Available: {device_available}")
print(f"Using Device: {device_name}")
DEVICE = "cuda" if device_available else "cpu"


In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
import subprocess, sys

packages = [
    'torch',
    'lettucedetect',
    'sentence-transformers',
    'transformers',
    'sentencepiece',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'requests',
    'datasets',
    'nltk',
]

print('Installing packages...')
for pkg in packages:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f'  ✅ {pkg}')
    except Exception as e:
        print(f'  ⚠️ {pkg} - install issue: {e}')

print('\n✅ Package installations complete!')


In [ ]:
# ============================================================
# ⚙️ NLTK Download & Setup
# ============================================================
import nltk
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    print("NLTK datasets checked successfully!")
except Exception as e:
    print(f"NLTK error: {e}")


In [ ]:
# ============================================================
# 📁 Load Datasets
# ============================================================
import urllib.request
import json
import pandas as pd
import traceback

print("Loading RAGTruth QA subset...")
from datasets import load_dataset
try:
    ds = load_dataset('wandb/RAGTruth-processed', split='test')
    qa_ds = ds.filter(lambda x: x['task_type'] == 'QA')
    
    ragtruth_subset = []
    for idx in range(min(50, len(qa_ds))):
        item = qa_ds[idx]
        lbl_dict = item['hallucination_labels_processed']
        has_hall = lbl_dict.get('evident_conflict', 0) > 0 or lbl_dict.get('baseless_info', 0) > 0
        label_str = "hallucinated" if has_hall else "faithful"
        ragtruth_subset.append({
            "id": f"RT_{idx:02d}",
            "query": item['query'],
            "retrieved_docs": [item['context']],
            "response": item['output'],
            "label": label_str
        })
    print(f"Successfully loaded {len(ragtruth_subset)} RAGTruth QA examples.")
except Exception as e:
    print(f"RAGTruth loading failed: {e}")
    ragtruth_subset = []

# Custom Mock Dataset modeling topically-related documents with wrong capabilities
mock_dataset = [
    {
        "id": "H01",
        "query": "Can I push/pull code via GitHub integration in Site24x7?",
        "retrieved_docs": [
            "Site24x7 GitHub Integration: Monitor your GitHub repositories health, "
            "track commit frequency, pull request metrics, and repository activity. "
            "Set up alerts for unusual patterns in your codebase activity."
        ],
        "response": (
            "Yes, Site24x7 supports full GitHub push/pull integration. You can push "
            "and pull code directly from the Site24x7 dashboard."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H02",
        "query": "Does Zoho CRM support video calling with customers?",
        "retrieved_docs": [
            "Zoho CRM Telephony Integration: Make and receive voice calls directly within "
            "Zoho CRM using built-in telephony features. Log call details automatically."
        ],
        "response": (
            "Yes, Zoho CRM supports video calling with customers through its telephony integration. "
            "You can make video calls directly from contact records."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H03",
        "query": "Can Zoho Desk automatically translate support tickets?",
        "retrieved_docs": [
            "Zoho Desk Multi-language Support: Create articles in multiple languages and "
            "configure language-specific email templates for the help center portal."
        ],
        "response": (
            "Yes, Zoho Desk automatically translates all incoming support tickets for agents."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H04",
        "query": "Does Zoho Books support cryptocurrency payments?",
        "retrieved_docs": [
            "Zoho Books Multi-Currency Support: Handle transactions in 170+ traditional "
            "currencies with automatic exchange rate updates."
        ],
        "response": (
            "Yes, Zoho Books supports cryptocurrency payments like Bitcoin and Ethereum."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H05",
        "query": "Can Zoho Projects automatically assign tasks using AI?",
        "retrieved_docs": [
            "Zoho Projects Task Management: Create tasks, deadlines, priorities, "
            "and assign tasks manually or in bulk to team members."
        ],
        "response": (
            "Yes, Zoho Projects uses built-in AI to assign tasks automatically based on workload."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H06",
        "query": "Does Zoho Analytics support real-time streaming data ingestion?",
        "retrieved_docs": [
            "Zoho Analytics Data Import: Import data from CSV, databases, and apps, "
            "and schedule automatic syncs at intervals ranging from 30 minutes to daily."
        ],
        "response": (
            "Yes, Zoho Analytics supports real-time streaming data ingestion through live streams."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H07",
        "query": "Can Zoho Mail compose AI-generated replies?",
        "retrieved_docs": [
            "Zoho Mail Smart Features: Use canned responses with pre-written templates "
            "organized by category to quickly reply to emails."
        ],
        "response": (
            "Yes, Zoho Mail includes a generative AI helper that writes replies for you."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H08",
        "query": "Does Site24x7 support fully automated server self-healing?",
        "retrieved_docs": [
            "Site24x7 IT Automation: Configure automation rules triggered by alerts, "
            "including restarting services and executing custom local scripts on servers."
        ],
        "response": (
            "Yes, Site24x7 detects issue root causes and performs fully autonomous self-healing."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H09",
        "query": "Can Zoho Creator deploy apps directly to the Apple App Store?",
        "retrieved_docs": [
            "Zoho Creator Mobile Access: Access Creator apps on iOS and Android via the "
            "official Zoho Creator app available on both mobile stores."
        ],
        "response": (
            "Yes, Zoho Creator can compile and publish apps directly to the Apple App Store."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H10",
        "query": "Does Zoho Desk support voice ticket creation with speech-to-text?",
        "retrieved_docs": [
            "Zoho Desk Multi-Channel Support: Create tickets from email, web forms, live chat, "
            "social media, and voice calls where agents log details manually."
        ],
        "response": (
            "Yes, customers can speak, and Zoho Desk automatically transcribes and logs tickets."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H11",
        "query": "Can Zoho Books automatically categorize expenses using AI?",
        "retrieved_docs": [
            "Zoho Books Expense Tracking: Upload receipt images, record expenses manually, "
            "and map them into pre-defined categories and projects."
        ],
        "response": (
            "Yes, the built-in AI automatically reads receipts and categorizes expenses for you."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H12",
        "query": "Can Zoho Analytics generate natural language insights from charts?",
        "retrieved_docs": [
            "Zoho Analytics Visualizations: Choose from 50+ interactive chart types, "
            "pivot tables, and dashboard builders with dynamic filters."
        ],
        "response": (
            "Yes, the analytics dashboard automatically writes a natural language summary chart."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H13",
        "query": "Does Zoho Projects support automatic time tracking based on app usage?",
        "retrieved_docs": [
            "Zoho Projects Time Tracking: Log task time manually or start/stop a timer "
            "directly from the task details page inside the project workspace."
        ],
        "response": (
            "Yes, it tracks background app usage to log timesheet records automatically."
        ),
        "label": "hallucinated"
    },
    {
        "id": "F01",
        "query": "Does Site24x7 support GitHub repository monitoring?",
        "retrieved_docs": [
            "Site24x7 GitHub Integration: Monitor your GitHub repositories health, "
            "track commit frequency, pull request metrics, and repository activity."
        ],
        "response": (
            "Yes, Site24x7 supports GitHub integration to track repository health, commits, and pull requests."
        ),
        "label": "faithful"
    },
    {
        "id": "F02",
        "query": "How do I set up multi-language support in Zoho Desk help center?",
        "retrieved_docs": [
            "Zoho Desk Multi-language Support: Create articles in multiple languages and "
            "configure language-specific email templates for the help center portal."
        ],
        "response": (
            "You can write article translations and set up customized templates for each language portal."
        ),
        "label": "faithful"
    },
    {
        "id": "F03",
        "query": "Can I handle foreign currency transactions in Zoho Books?",
        "retrieved_docs": [
            "Zoho Books Multi-Currency Support: Handle transactions in 170+ traditional "
            "currencies with automatic exchange rate updates."
        ],
        "response": (
            "Yes, Zoho Books lets you handle transactions across 170+ foreign currencies with rates synced."
        ),
        "label": "faithful"
    },
    {
        "id": "F04",
        "query": "What task management features are available in Zoho Projects?",
        "retrieved_docs": [
            "Zoho Projects Task Management: Create tasks, deadlines, priorities, "
            "and assign tasks manually or in bulk to team members."
        ],
        "response": (
            "It supports task creation, deadline setting, priority tagging, and manual or bulk assignments."
        ),
        "label": "faithful"
    },
    {
        "id": "F05",
        "query": "How does Zoho Analytics handle data imports?",
        "retrieved_docs": [
            "Zoho Analytics Data Import: Import data from CSV, databases, and apps, "
            "and schedule automatic syncs at intervals ranging from 30 minutes to daily."
        ],
        "response": (
            "You can import data from databases, files, or applications and sync them hourly or daily."
        ),
        "label": "faithful"
    },
    {
        "id": "F06",
        "query": "What are the auto-reply options in Zoho Mail?",
        "retrieved_docs": [
            "Zoho Mail Smart Features: Use canned responses with pre-written templates "
            "organized by category to quickly reply to emails."
        ],
        "response": (
            "You can use canned email templates organized by category to respond to messages."
        ),
        "label": "faithful"
    },
    {
        "id": "F07",
        "query": "What IT automation actions can I configure in Site24x7?",
        "retrieved_docs": [
            "Site24x7 IT Automation: Configure automation rules triggered by alerts, "
            "including restarting services and executing custom local scripts on servers."
        ],
        "response": (
            "You can set alerts to trigger automated tasks such as script execution and service restarts."
        ),
        "label": "faithful"
    },
    {
        "id": "F08",
        "query": "Can I access Zoho Creator apps on my phone?",
        "retrieved_docs": [
            "Zoho Creator Mobile Access: Access Creator apps on iOS and Android via the "
            "official Zoho Creator app available on both mobile stores."
        ],
        "response": (
            "Yes, you can run your customized apps by installing the Creator app from the app store."
        ),
        "label": "faithful"
    },
    {
        "id": "F09",
        "query": "What channels does Zoho Desk support for creating tickets?",
        "retrieved_docs": [
            "Zoho Desk Multi-Channel Support: Create tickets from email, web forms, live chat, "
            "social media, and voice calls where agents log details manually."
        ],
        "response": (
            "Tickets can be created from email, social media, voice calls, forms, or chat feeds."
        ),
        "label": "faithful"
    },
    {
        "id": "F10",
        "query": "How do I track expenses in Zoho Books?",
        "retrieved_docs": [
            "Zoho Books Expense Tracking: Upload receipt images, record expenses manually, "
            "and map them into pre-defined categories and projects."
        ],
        "response": (
            "Upload invoices/receipts, log costs manually, and map them directly into specific category tags."
        ),
        "label": "faithful"
    },
    {
        "id": "F11",
        "query": "How do I log time on tasks in Zoho Projects?",
        "retrieved_docs": [
            "Zoho Projects Time Tracking: Log task time manually or start/stop a timer "
            "directly from the task details page inside the project workspace."
        ],
        "response": (
            "You can log hours manually or run the task timer to populate sheets automatically."
        ),
        "label": "faithful"
    },
    {
        "id": "F12",
        "query": "What visualization options does Zoho Analytics offer?",
        "retrieved_docs": [
            "Zoho Analytics Visualizations: Choose from 50+ interactive chart types, "
            "pivot tables, and dashboard builders with dynamic filters."
        ],
        "response": (
            "It offers more than 50 dashboard chart styles along with pivot grids and filter options."
        ),
        "label": "faithful"
    }
]
print(f"Mock adjacent-capability dataset has {len(mock_dataset)} examples loaded.")


---
## ⚙️ Section 1: Model Initialization
We load LettuceDetect, HHEM, MiniCheck, and Relevance Embedders using safe memory/offload patches.

In [ ]:
# ============================================================
# 🧪 Monkey Patch for Safe CPU Memory Loading
# ============================================================
import transformers

transformers.PreTrainedModel.all_tied_weights_keys = property(
    lambda self: getattr(self, "_all_tied_weights_keys", {}),
    lambda self, v: setattr(self, "_all_tied_weights_keys", v)
)

orig_from_pretrained = transformers.PreTrainedModel.from_pretrained.__func__
transformers.PreTrainedModel.from_pretrained = classmethod(
    lambda cls, *args, **kwargs: orig_from_pretrained(
        cls, *args, **{**kwargs, "device_map": None} if "device_map" in kwargs else kwargs
    )
)

print("Monkey patches applied successfully!")


In [ ]:
# ============================================================
# 🧠 Load Models & Embedders
# ============================================================
from sentence_transformers import SentenceTransformer
from lettucedetect.models.inference import HallucinationDetector
from transformers import AutoModelForSequenceClassification
from minicheck.minicheck import MiniCheck

print("Loading Relevance embedder (all-MiniLM-L6-v2)...")
relevance_embedder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

print("Loading LettuceDetect (base-modernbert)...")
lettuce_detector = HallucinationDetector(
    method="transformer",
    model_path="KRLabsOrg/lettucedect-base-modernbert-en-v1"
)

print("Loading Vectara HHEM-2.1-Open...")
hhem_model = AutoModelForSequenceClassification.from_pretrained(
    "vectara/hallucination_evaluation_model", trust_remote_code=True
).to(DEVICE)
with torch.no_grad():
    hhem_model.t5.transformer.encoder.embed_tokens.weight.data.copy_(
        hhem_model.t5.transformer.shared.weight.data
    )

print("Loading MiniCheck (flan-t5-large)...")
minicheck_scorer = MiniCheck(model_name='flan-t5-large')

print("All evaluation models successfully loaded into RAM!")


---
## 🤖 Section 2: Zero-Shot vs Few-Shot LLM judges (Ollama)
We implement the local Llama 3.2 judges.

In [ ]:
# ============================================================
# 💬 Ollama Judge Implementation
# ============================================================
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:latest"

def query_ollama(prompt):
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {"temperature": 0.0}
            },
            timeout=20
        )
        if resp.status_code == 200:
            val = resp.json().get("response", "").strip()
            parsed = json.loads(val)
            return parsed.get("verdict", "error"), parsed.get("reason", "")
    except Exception as e:
        pass
    return "error", "Ollama API connectivity error"

def run_zero_shot_judge(query, docs, response):
    docs_joined = "\n\n".join(docs)
    prompt = (
        "You are a fact-checker.\n\n"
        f"Question: {query}\n\n"
        f"Retrieved Documents:\n{docs_joined}\n\n"
        f"Answer to verify:\n{response}\n\n"
        "Determine if the answer is fully supported by the documents. "
        "Respond ONLY with a JSON dictionary matching this schema:\n"
        '{"verdict": "faithful" or "hallucinated", "reason": "short explanation"}'
    )
    return query_ollama(prompt)

def run_few_shot_judge(query, docs, response):
    docs_joined = "\n\n".join(docs)
    prompt = (
        "You are a strict technical fact-checker for a RAG-based support system.\n"
        "Instruction: A document merely mentioning the same general topic (e.g. 'GitHub') "
        "does NOT support a specific claimed capability (e.g. 'push/pull integration'). "
        "Only mark the answer as faithful if the exact claim is explicitly stated in the context.\n\n"
        "=== Worked Example 1 (HALLUCINATION) ===\n"
        "Question: Can I push/pull code via Site24x7 GitHub integration?\n"
        "Document: 'Site24x7 GitHub Integration: Monitor repository health, commits, and pull requests.'\n"
        "Answer: Yes, you can push and pull code directly via the Site24x7 dashboard.\n"
        "Output: {\"verdict\": \"hallucinated\", \"reason\": \"Document mentions monitoring but not push/pull capability.\"}\n\n"
        "=== Worked Example 2 (FAITHFUL) ===\n"
        "Question: Does Site24x7 monitor GitHub?\n"
        "Document: 'Site24x7 GitHub Integration: Monitor repository health, commits, and pull requests.'\n"
        "Answer: Yes, Site24x7 supports GitHub integration to monitor repository health.\n"
        "Output: {\"verdict\": \"faithful\", \"reason\": \"Document explicitly confirms monitoring of repository health.\"}\n\n"
        "=== Active Evaluation Case ===\n"
        f"Question: {query}\n"
        f"Documents:\n{docs_joined}\n"
        f"Answer:\n{response}\n\n"
        "Respond ONLY with JSON format matching the schema above:\n"
        '{"verdict": "faithful" or "hallucinated", "reason": "short explanation"}'
    )
    return query_ollama(prompt)


---
## 🛠️ Section 3: Evaluation Logic & Tiered Pipeline
We implement the relevance checking, sentence-level MiniCheck parser, and the tiered/composite pipeline.

In [ ]:
from sentence_transformers import util
from nltk.tokenize import sent_tokenize
import time

RELEVANCE_THRESHOLD = 0.40

def run_composite_pipeline(query, retrieved_docs, response):
    # 1. Relevance check (Max Cosine Similarity)
    q_emb = relevance_embedder.encode(query, convert_to_tensor=True)
    doc_embs = relevance_embedder.encode(retrieved_docs, convert_to_tensor=True)
    cos_sims = util.cos_sim(q_emb, doc_embs)[0].cpu().tolist()
    max_relevance = max(cos_sims) if cos_sims else 0.0
    
    # 2. Split response into claims
    claims = sent_tokenize(response)
    if not claims:
        claims = [response]
        
    # 3. MiniCheck Sentence-Level Faithfulness & Borderline Escalation
    docs_combined = " ".join(retrieved_docs)
    minicheck_docs = [docs_combined] * len(claims)
    
    # MiniCheck score unpacking (4 values: pred_label, raw_prob, _, _)
    pred_labels, raw_probs, _, _ = minicheck_scorer.score(docs=minicheck_docs, claims=claims)
    
    final_verdicts = []
    escalated = False
    
    for label, prob, claim in zip(pred_labels, raw_probs, claims):
        # Escalation rule for borderline cases (prob between 0.30 and 0.70)
        if 0.30 <= prob <= 0.70:
            verdict, _ = run_few_shot_judge(query, retrieved_docs, claim)
            final_verdicts.append(verdict)
            escalated = True
        else:
            final_verdicts.append("faithful" if label == 1 else "hallucinated")
            
    # Composite verdict aggregation
    is_hallucinated = any(v == "hallucinated" for v in final_verdicts)
    final_label = "hallucinated" if is_hallucinated else "faithful"
    
    return {
        "predicted_label": final_label,
        "relevance": max_relevance,
        "escalated": escalated
    }


---
## 🩺 Section 4: Main Evaluation Loop
We evaluate each standalone tool and pipeline on both datasets, capturing latencies and errors explicitly.

In [ ]:
results = {
    "mock": [],
    "ragtruth": []
}

# Set up tracking stats
completion_stats = {
    "mock": {
        "LettuceDetect": {"attempted": 0, "valid": 0},
        "HHEM": {"attempted": 0, "valid": 0},
        "ZeroShot-LLM": {"attempted": 0, "valid": 0},
        "FewShot-LLM": {"attempted": 0, "valid": 0},
        "MiniCheck": {"attempted": 0, "valid": 0},
        "Composite": {"attempted": 0, "valid": 0}
    },
    "ragtruth": {
        "LettuceDetect": {"attempted": 0, "valid": 0},
        "HHEM": {"attempted": 0, "valid": 0},
        "ZeroShot-LLM": {"attempted": 0, "valid": 0},
        "FewShot-LLM": {"attempted": 0, "valid": 0},
        "MiniCheck": {"attempted": 0, "valid": 0},
        "Composite": {"attempted": 0, "valid": 0}
    }
}

last_exception = None

for dataset_name, dataset in [("mock", mock_dataset), ("ragtruth", ragtruth_subset)]:
    if not dataset: 
        continue
    print(f"\nEvaluating dataset: {dataset_name.upper()} ({len(dataset)} examples)")
    print("-" * 70)
    
    for i, row in enumerate(dataset):
        q = row["query"]
        docs = row["retrieved_docs"]
        resp = row["response"]
        gt = row["label"]
        row_id = row["id"]
        
        eval_item = {
            "id": row_id,
            "gt_label": gt,
            "predictions": {},
            "latencies": {}
        }
        
        # 1. LettuceDetect
        tool = "LettuceDetect"
        completion_stats[dataset_name][tool]["attempted"] += 1
        t0 = time.time()
        try:
            preds = lettuce_detector.predict(context=docs, question=q, answer=resp)
            ld_pred = "hallucinated" if any(preds) else "faithful"
            eval_item["predictions"][tool] = ld_pred
            eval_item["latencies"][tool] = time.time() - t0
            completion_stats[dataset_name][tool]["valid"] += 1
        except Exception as ex:
            print(f"Exception in {tool} on {dataset_name} ID {row_id}:")
            traceback.print_exc() if 'traceback' in globals() else print(ex)
            last_exception = (tool, row_id, ex)
            eval_item["predictions"][tool] = "error"
            
        # 2. Vectara HHEM-2.1-Open
        tool = "HHEM"
        completion_stats[dataset_name][tool]["attempted"] += 1
        t0 = time.time()
        try:
            docs_joined = " ".join(docs)
            hhem_score = float(hhem_model.predict([(docs_joined, resp)])[0].item())
            # Threshold calibrates scores > 0.5 as faithful, else hallucinated
            eval_item["predictions"][tool] = "faithful" if hhem_score > 0.5 else "hallucinated"
            eval_item["latencies"][tool] = time.time() - t0
            completion_stats[dataset_name][tool]["valid"] += 1
        except Exception as ex:
            print(f"Exception in {tool} on {dataset_name} ID {row_id}:")
            traceback.print_exc() if 'traceback' in globals() else print(ex)
            last_exception = (tool, row_id, ex)
            eval_item["predictions"][tool] = "error"
            
        # 3. Zero-Shot LLM Judge
        tool = "ZeroShot-LLM"
        completion_stats[dataset_name][tool]["attempted"] += 1
        t0 = time.time()
        try:
            verdict, _ = run_zero_shot_judge(q, docs, resp)
            eval_item["predictions"][tool] = verdict
            eval_item["latencies"][tool] = time.time() - t0
            completion_stats[dataset_name][tool]["valid"] += 1
        except Exception as ex:
            print(f"Exception in {tool} on {dataset_name} ID {row_id}:")
            traceback.print_exc() if 'traceback' in globals() else print(ex)
            last_exception = (tool, row_id, ex)
            eval_item["predictions"][tool] = "error"
            
        # 4. Few-Shot LLM Judge
        tool = "FewShot-LLM"
        completion_stats[dataset_name][tool]["attempted"] += 1
        t0 = time.time()
        try:
            verdict, _ = run_few_shot_judge(q, docs, resp)
            eval_item["predictions"][tool] = verdict
            eval_item["latencies"][tool] = time.time() - t0
            completion_stats[dataset_name][tool]["valid"] += 1
        except Exception as ex:
            print(f"Exception in {tool} on {dataset_name} ID {row_id}:")
            traceback.print_exc() if 'traceback' in globals() else print(ex)
            last_exception = (tool, row_id, ex)
            eval_item["predictions"][tool] = "error"
            
        # 5. MiniCheck
        tool = "MiniCheck"
        completion_stats[dataset_name][tool]["attempted"] += 1
        t0 = time.time()
        try:
            claims = sent_tokenize(resp)
            if not claims: claims = [resp]
            combined = " ".join(docs)
            # Correct usage unpacking signature of MiniCheck API
            pred_labels, raw_probs, _, _ = minicheck_scorer.score(docs=[combined]*len(claims), claims=claims)
            mc_pred = "faithful" if all(l == 1 for l in pred_labels) else "hallucinated"
            eval_item["predictions"][tool] = mc_pred
            eval_item["latencies"][tool] = time.time() - t0
            completion_stats[dataset_name][tool]["valid"] += 1
        except Exception as ex:
            print(f"Exception in {tool} on {dataset_name} ID {row_id}:")
            traceback.print_exc() if 'traceback' in globals() else print(ex)
            last_exception = (tool, row_id, ex)
            eval_item["predictions"][tool] = "error"
            
        # 6. Composite Pipeline
        tool = "Composite"
        completion_stats[dataset_name][tool]["attempted"] += 1
        t0 = time.time()
        try:
            comp_result = run_composite_pipeline(q, docs, resp)
            eval_item["predictions"][tool] = comp_result["predicted_label"]
            eval_item["latencies"][tool] = time.time() - t0
            eval_item["composite_escalated"] = comp_result["escalated"]
            completion_stats[dataset_name][tool]["valid"] += 1
        except Exception as ex:
            print(f"Exception in {tool} on {dataset_name} ID {row_id}:")
            traceback.print_exc() if 'traceback' in globals() else print(ex)
            last_exception = (tool, row_id, ex)
            eval_item["predictions"][tool] = "error"
            
        results[dataset_name].append(eval_item)
        
    print(f"Finished evaluating dataset: {dataset_name.upper()}")


---
## 🚦 Section 5: Mandatory Dry Run Check
Verify that all loops executed cleanly without error thresholds being tripped.

In [ ]:
# ============================================================
# 🚦 Mandatory Dry Run Verification
# ============================================================
print("=== Completion Diagnostic Check ===")
error_tripped = False

for dname in ["mock", "ragtruth"]:
    print(f"\nDataset: {dname.upper()}")
    for tool_name, stats in completion_stats[dname].items():
        attempted = stats["attempted"]
        valid = stats["valid"]
        print(f"  - {tool_name:15s}: {valid}/{attempted} successful predictions")
        if attempted - valid > 0:
            error_tripped = True

if error_tripped:
    print("\n❌ DRY RUN FAILURE: Errors encountered during evaluation loop!")
    if last_exception:
        print(f"First failure trace on tool '{last_exception[0]}' (ID {last_exception[1]}):")
        raise last_exception[2]
else:
    print("\n✅ DRY RUN SUCCESS: All tools completed successfully across all datasets!")


---
## 📊 Section 6: Compute Metrics & Tabulate
Calculate Precision, Recall, F1, Accuracy, Latencies, and Escalation metrics.

In [ ]:
# ============================================================
# 📐 Metrics Computation & Reporting
# ============================================================
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

metrics_list = []

for dname, label in [("mock", "Mock (Adjacent-Cap.)"), ("ragtruth", "RAGTruth QA Subset")]:
    dataset_results = results[dname]
    y_true = [x["gt_label"] for x in dataset_results]
    y_true_bin = [1 if t == "hallucinated" else 0 for t in y_true]
    
    for tool in ["LettuceDetect", "HHEM", "ZeroShot-LLM", "FewShot-LLM", "MiniCheck", "Composite"]:
        y_pred = [x["predictions"][tool] for x in dataset_results]
        y_pred_bin = [1 if p == "hallucinated" else 0 for p in y_pred]
        
        # Calculate latency averages
        latencies = [x["latencies"][tool] for x in dataset_results if tool in x["latencies"]]
        avg_latency = sum(latencies) / len(latencies) if latencies else 0.0
        
        p = precision_score(y_true_bin, y_pred_bin, zero_division=0)
        r = recall_score(y_true_bin, y_pred_bin, zero_division=0)
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        acc = accuracy_score(y_true_bin, y_pred_bin)
        
        # Escalation rate tracking for composite pipeline
        esc_rate_str = "N/A"
        if tool == "Composite":
            esc_list = [1 if x.get("composite_escalated", False) else 0 for x in dataset_results]
            esc_rate = sum(esc_list) / len(esc_list) if esc_list else 0.0
            esc_rate_str = f"{esc_rate * 100:.1f}%"
            
        metrics_list.append({
            "tool": tool,
            "dataset": label,
            "precision": p,
            "recall": r,
            "f1": f1,
            "accuracy": acc,
            "avg_latency": avg_latency,
            "escalation_rate": esc_rate_str
        })

df_metrics = pd.DataFrame(metrics_list)

print("=" * 95)
print("📊  EVALUATION METRICS TABLE")
print("=" * 95)
print(df_metrics.to_string(index=False))


---
## 📈 Section 7: Visualizations & Confusion Matrices
Save visual performance plots and compute confusion matrices for mock dataset.

In [ ]:
# ============================================================
# 📈 Visual charts & Confusion Matrices
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrices on Mock dataset
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

dataset_results = results["mock"]
y_true = [x["gt_label"] for x in dataset_results]
y_true_bin = [1 if t == "hallucinated" else 0 for t in y_true]
tools = ["LettuceDetect", "HHEM", "ZeroShot-LLM", "FewShot-LLM", "MiniCheck", "Composite"]

for idx, tool in enumerate(tools):
    y_pred = [x["predictions"][tool] for x in dataset_results]
    y_pred_bin = [1 if p == "hallucinated" else 0 for p in y_pred]
    cm = confusion_matrix(y_true_bin, y_pred_bin)
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=["Faithful", "Hallucinated"],
                yticklabels=["Faithful", "Hallucinated"])
    axes[idx].set_title(f"{tool} (Mock)")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrices saved to confusion_matrices.png!")


---
## 🏆 Section 8: Summary and Trade-Off Analysis
We present the final analysis and trade-offs.

In [ ]:
# ============================================================
# 🏆 Summary & Trade-off Cell
# ============================================================
print("=" * 80)
print("🏆  FINAL ANALYSIS AND RECOMMENDATIONS")
print("=" * 80)

for label in ["Mock (Adjacent-Cap.)", "RAGTruth QA Subset"]:
    sub_df = df_metrics[df_metrics["dataset"] == label]
    best_acc = sub_df.loc[sub_df["accuracy"].idxmax()]
    best_f1 = sub_df.loc[sub_df["f1"].idxmax()]
    fastest = sub_df.loc[sub_df["avg_latency"].idxmin()]
    
    print(f"\n=== Dataset: {label} ===")
    print(f"  🥇 Best Accuracy : {best_acc['tool']} (Acc: {best_acc['accuracy']:.3f})")
    print(f"  🥇 Best F1-Score : {best_f1['tool']} (F1: {best_f1['f1']:.3f})")
    print(f"  ⚡ Fastest Tool   : {fastest['tool']} (Avg Latency: {fastest['avg_latency']:.3f}s)")

print("\nDone!")
